# B11 · Making a beautiful map: a finished landslide map of Nepal

> **Time:** about 2 hours  
> **You need:** Part A (especially A10 · Plotting with Matplotlib) and Lessons B1 to B10, especially counting points per district (B9) and classes and legends (B10).  
> **By the end you can:**
> 1. plan a map for a purpose and an audience, and draw its layers in the right order,
> 2. add readable labels, a scale bar, a north arrow, a legend for every layer and an inset locator map,
> 3. remove clutter, credit the sources and save the map at 300 dpi for print.

Every lesson so far ended with a quick map to check a result. This lesson makes a map for **other people**: a finished, honest map of the practice landslide records that you could put in a report. Each step adds one element. Step 8 puts them all together in one cell, and you will end with a 300 dpi image file. The landslide records are made-up practice data, and the finished map says so.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 21 of 21 · [Course home](../README.md)  
← Previous: [B10 · Counting points and density maps: where are the landslides?](B10_Counting_points_and_density_maps.ipynb) · You have reached the end of the course

## 1. Plan before you plot

A good map starts on paper, not in code. Answer four questions first:

| Question | Our answer |
| --- | --- |
| **Purpose:** what should the reader learn? | Where the practice landslide records are, and which districts have most |
| **Audience:** who reads it? | Colleagues in a district office, with no GIS training |
| **Medium:** where will it appear? | A printed A4 report page, so 300 dpi and no interactivity |
| **Elements:** what must be on it? | Title, map, legend, scale bar, north arrow, locator inset, source note |

The main map needs a scale bar, which needs metres. So every layer of the main map goes into **UTM 45N** (EPSG:32645), as in B8 to B10. The data preparation below is the B9 counting recipe.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import mapclassify
from pathlib import Path

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

districts_utm = gpd.read_file(DATA / "nepal_districts.geojson").to_crs(32645)
provinces_utm = gpd.read_file(DATA / "nepal_provinces.geojson").to_crs(32645)
world = gpd.read_file(DATA / "world_countries.geojson")            # stays in degrees, for the inset

slides = pd.read_csv(DATA / "practice_landslides.csv")
slides["district"] = slides["district"].replace("Chitwan", "Chitawan")   # as in B6
landslides_utm = gpd.GeoDataFrame(
    slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326"
).to_crs(32645)

joined = gpd.sjoin(landslides_utm, districts_utm, how="inner", predicate="within")
counts = joined.groupby("district_code").size().reset_index(name="n_landslides")
district_counts = districts_utm.merge(counts, on="district_code", how="left")
district_counts["n_landslides"] = district_counts["n_landslides"].fillna(0).astype(int)
nepal_utm = districts_utm.dissolve()                                 # one outline for the whole country

print(district_counts["n_landslides"].sum(), len(district_counts))
print(districts_utm.crs == provinces_utm.crs == landslides_utm.crs)

All 40 records are counted in the 77 districts, and the three main layers share one CRS (`True`).

`dissolve()` without a column merges every district into one shape (B6). It gives the national outline, drawn a little bolder later.

### Practice

**Q1 · one line:** Print the name of the CRS of `district_counts`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(district_counts.crs.name)
```

`WGS 84 / UTM zone 45N`. Checking the CRS before drawing anything saves you from a wrong scale bar later.
</details>

**Q2 · two lines:** How large is the map area? Use `.total_bounds` of `nepal_utm` and print its width and height in km, rounded.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
xmin, ymin, xmax, ymax = nepal_utm.total_bounds
print(round((xmax - xmin) / 1000), round((ymax - ymin) / 1000))
```

About 796 km wide and 466 km tall, a ratio of roughly 1.7 to 1. A wide figure such as `figsize=(12, 7.5)` fits Nepal with room for the legend.
</details>

**Q3 · three lines:** Draw a first, plain draft: `district_counts` coloured by `n_landslides`, with the points on top. As in A10, lines that only set up the plot (`fig, ax = plt.subplots()` and `plt.show()`) do not count toward the line count. Then write down three things that a reader would miss on this draft.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, column="n_landslides", legend=True)
landslides_utm.plot(ax=ax, color="black", markersize=5)
plt.show()
```

A draft like this answers your own question, but a reader would miss a title, units and classes in the legend, a scale, and any note that the data are made up. The default colours also make 0 look like real data. The next steps fix each of these.
</details>

## 2. Layer order: what goes on top

Matplotlib paints like a person with a brush: each new layer covers what is already there. So draw from the **bottom up**:

1. filled polygons (districts),
2. boundary lines (provinces, the national outline), with `facecolor="none"` so they do not cover the fill,
3. points (landslides),
4. text (labels, title).

When the drawing order is hard to control, `zorder=` sets the order directly. Higher numbers are drawn on top.

**Predict:** in the left map below the districts are drawn after the points. What will you see?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

landslides_utm.plot(ax=ax1, color="black", markersize=10)
district_counts.plot(ax=ax1, color="tan", edgecolor="white", linewidth=0.3)     # drawn last: covers the points
ax1.set_title("Wrong order: points hidden")

district_counts.plot(ax=ax2, color="tan", edgecolor="white", linewidth=0.3)
provinces_utm.plot(ax=ax2, facecolor="none", edgecolor="black", linewidth=0.8)
landslides_utm.plot(ax=ax2, color="black", markersize=10)
ax2.set_title("Right order: fill, lines, points")

ax1.set_axis_off()
ax2.set_axis_off()
plt.show()

In the left map the landslides are gone: the districts were painted over them. In the right map, the fill sits at the bottom, the province lines above it and the points on top.

`facecolor="none"` draws only the outline of a polygon. Without it, the provinces would cover the districts.

### Practice

**Q1 · one line:** Plot only the outlines of the provinces, in dark grey, with no fill.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces_utm.plot(facecolor="none", edgecolor="dimgrey")
```

Seven outlines appear. An outline layer like this goes on top of the fill and below the points.
</details>

**Q2 · two lines:** Repair the wrong order **without** changing the order of the lines: draw the points first, but give them `zorder=3` so they end up on top.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
landslides_utm.plot(ax=ax, color="black", markersize=10, zorder=3)
district_counts.plot(ax=ax, color="tan", edgecolor="white", linewidth=0.3)
plt.show()
```

The points are visible again. Their `zorder` of 3 is higher than the polygons' default, so Matplotlib draws them last.
</details>

**Q3 · three lines:** Draw the districts in light grey, then `nepal_utm` as a bold outline (`linewidth=1.5`, no fill), then the points. Remove the axes.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, color="whitesmoke", edgecolor="grey", linewidth=0.3)
nepal_utm.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.5)
landslides_utm.plot(ax=ax, color="crimson", markersize=10)
ax.set_axis_off()
plt.show()
```

A bold national border frames the map and separates Nepal from the empty page. Line width is a quiet way to rank boundaries: nation > province > district.
</details>

## 3. Colours chosen on purpose

Colours should carry meaning, not decoration:

- **More** should look **darker**. Use one sequential run of colours, light to dark, as in B10.
- **Zero** gets a quiet neutral colour, so the eye skips it.
- **Rainbow** colour maps have no natural order and confuse readers (A10). Avoid them.
- Use the **same** colours for the same meaning across all your maps.

You can name exact colours with **hex codes** such as `"#d7301f"`: red, green and blue amounts in hexadecimal. Below, mapclassify's `UserDefined` classes (B10) give each district a class number in `.yb`. A list comprehension (A3) turns each class number into its colour, and `color=` paints every district with its own colour.

In [ ]:
colours = ["#efece6", "#fdcc8a", "#fc8d59", "#d7301f"]       # neutral, then light to dark orange-red
class_labels = ["No records", "1 to 2", "3 to 5", "6 to 9"]

classes = mapclassify.UserDefined(district_counts["n_landslides"], bins=[0, 2, 5, 9])
district_counts["colour"] = [colours[c] for c in classes.yb]
print(classes.counts)

fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, color=district_counts["colour"], edgecolor="white", linewidth=0.5)
ax.set_axis_off()
plt.show()

The four classes hold 69, 1, 5 and 2 districts. The 69 districts without records fade into a warm grey, and the eye goes straight to the two darkest districts, Chitawan and Dolakha.

White district borders look lighter than black ones and keep attention on the colours.

### Practice

**Q1 · one line:** Print the colour that Dolakha received.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(district_counts.loc[district_counts["district"] == "Dolakha", "colour"].iloc[0])
```

`#d7301f`, the darkest red: Dolakha has 9 records, in the class 6 to 9.
</details>

**Q2 · two lines:** Try a rainbow on purpose: plot `n_landslides` with `cmap="jet"` and a legend. Compare it with the map above. Which one tells you faster where most records are?

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, column="n_landslides", cmap="jet", legend=True)
plt.show()
```

With `jet`, zero is a strong dark blue that fills most of Nepal, and the middle values (3 to 5) glow bright cyan and green, as loud as the maximum. The eye is pulled to the wrong places. The sequential map answers the question at a glance.
</details>

**Q3 · three lines:** Make a second colour scheme in blues for the same classes: `["#eeeeee", "#bdd7e7", "#6baed6", "#2171b5"]`. Store the colours in a new column `blue` and draw the map with it.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
blues = ["#eeeeee", "#bdd7e7", "#6baed6", "#2171b5"]
district_counts["blue"] = [blues[c] for c in classes.yb]
fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, color=district_counts["blue"], edgecolor="white", linewidth=0.5)
plt.show()
```

The same story in a calmer colour. Keeping the colours in a list makes a restyle a one-line change, and lets you reuse the same colours in every map of a report.
</details>

## 4. Labels that can be read

Label only what the reader needs: here, the 8 districts with records. Labelling all 77 would bury the message.

Place each label at `.representative_point()`, a point guaranteed to lie **inside** the polygon. A centroid can fall outside a curved or hollow shape.

`ax.annotate(text, xy=(x, y), xytext=(0, 9), textcoords="offset points")` writes the text 9 points above the spot, so it does not sit on top of a landslide marker. A **halo**, a thin white outline around each letter, keeps text readable over any colour. Matplotlib makes it with `path_effects`.

`for _, row in table.iterrows():` loops over the rows of a table, one row (a Series) at a time. The `_` is a name for the row label you do not need. A dictionary with `.get(key, default)` (A4) **nudges** a few labels that would collide.

In [ ]:
import matplotlib.patheffects as pe

halo = [pe.withStroke(linewidth=3, foreground="white")]
nudge = {"Sindhupalchok": (-8, -16), "Chitawan": (0, -18)}     # labels moved below their district
with_records = district_counts[district_counts["n_landslides"] > 0]

fig, ax = plt.subplots(figsize=(12, 7))
district_counts.plot(ax=ax, color=district_counts["colour"], edgecolor="white", linewidth=0.5)
landslides_utm.plot(ax=ax, color="#1a1a1a", edgecolor="white", linewidth=0.5, markersize=22)
for _, row in with_records.iterrows():
    spot = row.geometry.representative_point()
    ax.annotate(row["district"], xy=(spot.x, spot.y), xytext=nudge.get(row["district"], (0, 9)),
                textcoords="offset points", ha="center", fontsize=8, fontweight="bold",
                path_effects=halo)
ax.set_axis_off()
plt.show()

Eight bold labels sit above their districts, with white halos. Sindhupalchok and Chitawan are nudged below, because above they would collide with Dolakha's label and Chitawan's points.

Nudging by hand is normal cartography. Even professional map software leaves the last few labels to a person.

### Practice

**Q1 · one line:** Show why `representative_point()` is safer. Make a thin crescent with `from shapely.geometry import Point` and `moon = Point(0, 0).buffer(10).difference(Point(4, 0).buffer(9))`, then print whether its centroid and its representative point lie within it.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
from shapely.geometry import Point
moon = Point(0, 0).buffer(10).difference(Point(4, 0).buffer(9))
print(moon.centroid.within(moon), moon.representative_point().within(moon))
```

`False True`: the crescent's centroid falls in the empty hollow, but the representative point is inside by definition. Nepal's districts happen to be compact enough, but valleys and lakes are not.
</details>

**Q2 · two lines:** Label the 7 provinces of `provinces_utm` with their codes, at their representative points. Draw the province outlines first.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))
provinces_utm.plot(ax=ax, facecolor="whitesmoke", edgecolor="grey")
for _, row in provinces_utm.iterrows():
    spot = row.geometry.representative_point()
    ax.annotate(row["province_code"], xy=(spot.x, spot.y), ha="center")
plt.show()
```

Seven codes, NP01 to NP07, each inside its province. Without `xytext`, the text sits exactly on the point.
</details>

**Q3 · three lines:** Write labels that show the count too, such as `Dolakha (9)`, with an f-string (A2). Print the 8 label texts by looping over `with_records`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for _, row in with_records.iterrows():
    label = f"{row['district']} ({row['n_landslides']})"
    print(label)
```

Eight lines such as `Chitawan (9)` and `Kaski (1)`. A count in the label lets readers read exact values without decoding the colours. Inside an f-string in double quotes, use single quotes around column names.
</details>

## 5. Scale bar and north arrow

A **scale bar** shows distance on the map. `ScaleBar(dx=1, units="m")` from `matplotlib_scalebar` says: one map unit is one metre. It then picks a round length, such as 100 km.

This works only in a **projected** CRS in metres, like UTM 45N. In degrees, a unit is not a fixed distance (B1, B4), so a scale bar on a degree map would lie.

A **north arrow** can be drawn with `ax.annotate("N", xy=..., xytext=..., arrowprops=...)`. `xycoords="axes fraction"` places it by fractions of the map box: (0, 0) is the lower left corner and (1, 1) the upper right.

In UTM 45N, the map's "up" is exactly true north only along 87°E, the zone's central meridian. Further west, true north tilts a little. **pyproj** can tell you by how much.

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar

fig, ax = plt.subplots(figsize=(12, 7))
district_counts.plot(ax=ax, color=district_counts["colour"], edgecolor="white", linewidth=0.5)
ax.add_artist(ScaleBar(dx=1, units="m", location="lower right", box_alpha=0, length_fraction=0.2))
ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
            ha="center", va="center", fontsize=14, fontweight="bold",
            arrowprops={"facecolor": "black", "width": 3, "headwidth": 10})
ax.set_axis_off()
plt.show()

In [ ]:
import pyproj

utm45 = pyproj.Proj("EPSG:32645")
for lon in [80.1, 84.0, 87.0, 88.1]:
    tilt = utm45.get_factors(lon, 28.0).meridian_convergence
    print(f"{lon}°E: true north differs from map up by {tilt:.1f}°")

The scale bar shows 150 km, and the arrow points to the top of the map. `length_fraction=0.2` asks for a bar about a fifth of the map's width; `box_alpha=0` removes its white box.

At 87°E the difference is 0° (printed as −0.0°), and at 80.1°E it is about −3.3°. On a national map, a vertical north arrow is fine; for compass work on the ground in far-west Nepal, that tilt matters.

### Practice

**Q1 · one line:** Add a scale bar in the **lower left** corner to a plain map of `provinces_utm`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
provinces_utm.plot(ax=ax, color="whitesmoke", edgecolor="grey")
ax.add_artist(ScaleBar(dx=1, units="m", location="lower left"))
plt.show()
```

The bar sits in the lower left with its default white box. `location` accepts the same corner names as Matplotlib legends.
</details>

**Q2 · two lines:** See the lie for yourself: plot the districts in **degrees** (`districts_utm.to_crs(4326)`) and add `ScaleBar(dx=1, units="m")`. Read the bar's label.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
districts_utm.to_crs(4326).plot(ax=ax, color="whitesmoke", edgecolor="grey")
ax.add_artist(ScaleBar(dx=1, units="m", location="lower right"))
plt.show()
```

The bar reads **1 m** for a length of about one degree, which on the ground is roughly 100 km (B1). It treats each degree as one metre. The only message is a warning about the aspect ratio, not about units: the library cannot know your units. That is why the main map must be projected.
</details>

**Q3 · three lines:** Print the tilt between true north and map up at Kathmandu (85.3°E, 27.7°N) and at Mahendranagar (80.2°E, 29.0°N), rounded to 1 decimal. Use `utm45` from the example.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for lon, lat in [(85.3, 27.7), (80.2, 29.0)]:
    tilt = utm45.get_factors(lon, lat).meridian_convergence
    print(lon, lat, round(tilt, 1))
```

About −0.8° at Kathmandu and −3.3° at Mahendranagar. The minus sign shows the direction of the tilt. The further from 87°E, the larger it gets.
</details>

## 6. One legend for every layer

When you colour districts with your own `color=` list, GeoPandas cannot make the legend for you. You build it yourself, which also lets one legend explain **every** layer.

A legend is a list of **handles**: small sample symbols, each with a label.

- `Patch(facecolor=..., label=...)` from `matplotlib.patches` is a coloured box, for areas.
- `Line2D([], [], marker="o", linestyle="none", ...)` from `matplotlib.lines` is a marker without a line, for points.

`ax.legend(handles=handles, title=..., loc=..., frameon=False)` draws them. `zip(colours, class_labels)` (B10) pairs each colour with its label.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

handles = [Patch(facecolor=c, edgecolor="#999999", label=lab) for c, lab in zip(colours, class_labels)]
handles.append(Line2D([], [], marker="o", linestyle="none", markerfacecolor="#1a1a1a",
                      markeredgecolor="white", markersize=7, label="Landslide record"))
print(len(handles), "legend entries")

fig, ax = plt.subplots(figsize=(12, 7))
district_counts.plot(ax=ax, color=district_counts["colour"], edgecolor="white", linewidth=0.5)
landslides_utm.plot(ax=ax, color="#1a1a1a", edgecolor="white", linewidth=0.5, markersize=22)
ax.legend(handles=handles, title="Landslide records per district", loc="lower left",
          frameon=False, fontsize=9, title_fontsize=10)
ax.set_axis_off()
plt.show()

The legend has 5 entries: four classes and the point symbol. Its symbols match the map exactly, because they use the same colour list.

The title states the quantity and the unit (records per district), and the classes use plain words. Southwest Nepal leaves an empty corner at the lower left, a natural place for the legend.

### Practice

**Q1 · one line:** Make a single legend handle for the province boundary lines: a `Line2D` with `color="#6b6b6b"`, `linewidth=1`, and the label `"Province boundary"`. Print it.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(Line2D([], [], color="#6b6b6b", linewidth=1, label="Province boundary"))
```

Matplotlib prints `Line2D(Province boundary)`. A `Line2D` without a marker is a short line, the right symbol for a boundary.
</details>

**Q2 · two lines:** Build a legend for the trigger types: one `Line2D` marker per trigger, using the colours `{"rainfall": "tab:blue", "road cut": "tab:orange", "earthquake": "tab:red"}`. Print the labels of the handles.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
trigger_colours = {"rainfall": "tab:blue", "road cut": "tab:orange", "earthquake": "tab:red"}
print([Line2D([], [], marker="o", linestyle="none", color=col, label=t).get_label() for t, col in trigger_colours.items()])
```

`['rainfall', 'road cut', 'earthquake']`. Looping over a dictionary's items (A4) keeps each label next to its colour, so they can never get out of step.
</details>

**Q3 · three lines:** Draw the map with the landslides coloured by trigger, using `trigger_colours` from Q2, and a matching hand-made legend. Hint: loop over the dictionary, plot the matching points, and build the handles in the same loop.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, color="whitesmoke", edgecolor="grey", linewidth=0.3)
handles_t = []
for t, col in trigger_colours.items():
    landslides_utm[landslides_utm["trigger"] == t].plot(ax=ax, color=col, markersize=15)
    handles_t.append(Line2D([], [], marker="o", linestyle="none", color=col, label=t))
ax.legend(handles=handles_t, title="Trigger", loc="lower left")
plt.show()
```

Plotting and legend come from the same dictionary, so colours and labels always match. This pattern works for any categorical layer.
</details>

## 7. An inset locator map

Readers abroad may not know where Nepal is. A small **locator map** in a corner answers that.

`ax.inset_axes([x0, y0, width, height])` makes a small map box inside the main one, in axes fractions (0 to 1). `[0.77, 0.70, 0.23, 0.30]` is the upper right corner, where northeast of Nepal the page is empty.

The inset is a separate map, so it may use another CRS. We draw `world_countries.geojson` in degrees and highlight Nepal. It needs no scale bar or north arrow: its only job is "you are here".

`set_xlim` and `set_ylim` zoom the inset to a region. `set_xticks([])` and `set_yticks([])` remove the ticks but keep the frame.

In [ ]:
print(world.crs, len(world))

fig, ax = plt.subplots(figsize=(12, 7))
district_counts.plot(ax=ax, color=district_counts["colour"], edgecolor="white", linewidth=0.5)

inset = ax.inset_axes([0.77, 0.70, 0.23, 0.30])
inset.set_facecolor("#e8f1f8")                                   # pale blue sea
world.plot(ax=inset, color="#dddddd", edgecolor="white", linewidth=0.3)
world[world["name"] == "Nepal"].plot(ax=inset, color="#d7301f")
inset.set_xlim(60, 110)
inset.set_ylim(5, 45)
inset.set_xticks([])
inset.set_yticks([])
inset.set_xlabel("")
inset.set_ylabel("")
inset.set_title("Nepal in Asia", fontsize=8)

ax.set_axis_off()
plt.show()

The world layer has 177 countries in EPSG:4326. In the inset, Nepal is a small red strip between the plains of India and the Tibetan Plateau, coloured with the darkest map colour so the two maps feel connected.

GeoPandas labels degree axes automatically, so `set_xlabel("")` and `set_ylabel("")` clear those labels.

### Practice

**Q1 · one line:** Print the name and continent of Nepal in `world`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(world.loc[world["name"] == "Nepal", ["name", "continent"]])
```

Nepal is listed in Asia. Checking the exact spelling in `name` first avoids an empty selection.
</details>

**Q2 · two lines:** Make a locator map on its own: plot the countries of Asia (`continent == "Asia"`) in light grey and Nepal in red.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(8, 6))
world[world["continent"] == "Asia"].plot(ax=ax, color="lightgrey", edgecolor="white")
world[world["name"] == "Nepal"].plot(ax=ax, color="#d7301f")
plt.show()
```

Asia fills the map, and Nepal is a small red mark. The whole continent is too large a frame for a district map; the inset above zooms in on 60°E to 110°E.
</details>

**Q3 · three lines:** Put the inset in the **lower right** corner instead, and also outline India (`name == "India"`) in dark grey so readers see the neighbour.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(12, 7))
district_counts.plot(ax=ax, color=district_counts["colour"], edgecolor="white", linewidth=0.5)
inset = ax.inset_axes([0.77, 0.0, 0.23, 0.30])
world.plot(ax=inset, color="#dddddd")
world[world["name"] == "India"].plot(ax=inset, facecolor="none", edgecolor="dimgrey")
world[world["name"] == "Nepal"].plot(ax=inset, color="#d7301f")
inset.set_xlim(60, 110)
inset.set_ylim(5, 45)
inset.set_xticks([])
inset.set_yticks([])
ax.set_axis_off()
plt.show()
```

Only the first two numbers change the position. In the lower right the inset now covers part of Taplejung, a district with records: an inset must never hide data. Every element competes for the same empty corners, which is why the upper right is the better place.
</details>

## 8. Putting it together: less clutter, title, source and a 300 dpi file

Now combine everything in one cell. Finishing touches:

- **Remove clutter:** no axis ticks (`set_axis_off`), no frame around the legend, no extra lines.
- **Title:** says what and where, left-aligned like a headline.
- **Source note:** credits the data, says the records are made up, and names the projection.
- **Save** with `fig.savefig(path, dpi=300, bbox_inches="tight")` (A10). 300 dots per inch is the usual print quality. `bbox_inches="tight"` trims the empty margin.

`ax.set_ylim(ymin - 40000, ymax)` adds 40 km of empty space below Nepal, so the scale bar does not touch the border. `ax.text(0, -0.02, ..., transform=ax.transAxes)` places the note just below the map box, using the same axes fractions as the north arrow.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7.5))

# 1. fill, 2. lines, 3. points
district_counts.plot(ax=ax, color=district_counts["colour"], edgecolor="white", linewidth=0.5)
provinces_utm.plot(ax=ax, facecolor="none", edgecolor="#6b6b6b", linewidth=1.0)
nepal_utm.plot(ax=ax, facecolor="none", edgecolor="#333333", linewidth=1.3)
landslides_utm.plot(ax=ax, color="#1a1a1a", edgecolor="white", linewidth=0.5, markersize=22, zorder=5)

# 4. labels with halos, nudged where needed
for _, row in with_records.iterrows():
    spot = row.geometry.representative_point()
    ax.annotate(row["district"], xy=(spot.x, spot.y), xytext=nudge.get(row["district"], (0, 9)),
                textcoords="offset points", ha="center", fontsize=8, fontweight="bold",
                color="#222222", path_effects=halo, zorder=6)

# 5. legend, scale bar and north arrow, with a little room below the map for the bar
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin - 40000, ymax)
ax.legend(handles=handles, title="Landslide records per district", loc="lower left",
          frameon=False, fontsize=9, title_fontsize=10)
ax.add_artist(ScaleBar(dx=1, units="m", location="lower right", box_alpha=0, length_fraction=0.2))
ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
            ha="center", va="center", fontsize=14, fontweight="bold",
            arrowprops={"facecolor": "black", "width": 3, "headwidth": 10})

# 6. locator inset
inset = ax.inset_axes([0.77, 0.70, 0.23, 0.30])
inset.set_facecolor("#e8f1f8")
world.plot(ax=inset, color="#dddddd", edgecolor="white", linewidth=0.3)
world[world["name"] == "Nepal"].plot(ax=inset, color="#d7301f")
inset.set_xlim(60, 110)
inset.set_ylim(5, 45)
inset.set_xticks([])
inset.set_yticks([])
inset.set_xlabel("")
inset.set_ylabel("")
inset.set_title("Nepal in Asia", fontsize=8)

# 7. title, source note, no clutter
ax.set_title("Practice landslide records by district, Nepal", fontsize=15, loc="left", fontweight="bold")
ax.text(0, -0.02, "Landslide records are made-up practice data (40 points). "
        "Boundaries: geoJSON-Nepal (OCHA / Survey Department), simplified. "
        "Locator: Natural Earth.\nProjection: UTM zone 45N (EPSG:32645). Map: PyGILE course.",
        transform=ax.transAxes, va="top", fontsize=7.5, color="#555555")
ax.set_axis_off()

# 8. save for print, then show
map_path = OUT / "practice_landslide_map.png"
fig.savefig(map_path, dpi=300, bbox_inches="tight")
plt.show()
print(map_path.exists())

This is a finished map: one clear message, layers in order, every symbol explained, distances measurable, Nepal located, and the source and status of the data stated. `True` confirms that the file was written.

**Final checklist.** Before you share any map, check each line:

| ✔ | Question |
| --- | --- |
| ☐ | Does the title say **what** and **where**? |
| ☐ | Is every colour and symbol in the legend, with **units**? |
| ☐ | Are zero and "no data" shown honestly, and is the classification sensible (B10)? |
| ☐ | Is the main map in a **projected** CRS, so the scale bar is true? |
| ☐ | Is there a north arrow, or is it clear that north is up? |
| ☐ | Can every label be read, with no collisions? |
| ☐ | Is there a locator map if readers may not know the place? |
| ☐ | Does a source note credit the data and flag made-up or incomplete data? |
| ☐ | Is everything that does not help the message removed? |
| ☐ | Is the file saved at the right resolution for its medium (300 dpi for print)? |

### Practice

**Q1 · one line:** Print the size of the saved file in kilobytes. `map_path.stat().st_size` gives the size in bytes.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(map_path.stat().st_size / 1024))
```

Roughly 580 KB (the exact number varies a little between computers). A 12 × 7.5 inch figure at 300 dpi has about 3,600 × 2,250 pixels, sharp enough for print.
</details>

**Q2 · two lines:** Change the title to show the total number of records with an f-string, then save the figure again. Use `fig` and `ax` from the example.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
ax.set_title(f"Practice landslide records by district, Nepal ({district_counts['n_landslides'].sum()} records)", fontsize=15, loc="left", fontweight="bold")
fig.savefig(map_path, dpi=300, bbox_inches="tight")
```

The title now ends with "(40 records)". A figure stays editable after it is drawn, so small changes do not need the whole cell again. Taking numbers from the data keeps the title correct if the data change.
</details>

**Q3 · three lines:** Save a light version for a web page: the same `fig` at 100 dpi to `practice_landslide_map_web.png` in `OUT`. Print both file sizes in kilobytes.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
web_path = OUT / "practice_landslide_map_web.png"
fig.savefig(web_path, dpi=100, bbox_inches="tight")
print(round(map_path.stat().st_size / 1024), round(web_path.stat().st_size / 1024))
```

Roughly 580 KB against 145 KB: the 100 dpi file is about four times smaller. Choose the resolution for the medium: 300 dpi for print, about 100 to 150 dpi for screens.
</details>

## 9. When it breaks: saving to a folder that does not exist

The course folder is called `outputs`, with an **s**. Here the path says `output`. Run the cell.

In [ ]:
fig.savefig(Path("..") / "output" / "map.png", dpi=300)

The last line reads `FileNotFoundError: [Errno 2] No such file or directory: '../output/map.png'`.

You met `FileNotFoundError` in A7 when **reading** a missing file. When **saving**, it means the folder does not exist: Matplotlib creates the file, but never the folder.

The fix: use the `OUT` variable you defined once, after `OUT.mkdir(exist_ok=True)`. Typing a path once and reusing it avoids this whole family of errors.

**Try it:** change the cell to save into `OUT`.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines, not counting plot setup lines. Plan the map in words first (purpose, audience, elements), then write the code.

### Review 1 · A Bagmati map with every district named

Make a map of Bagmati Province (`province_code` `"NP03"`) from `district_counts`, using the `colour` column. Add the landslides inside Bagmati, a label with a white halo for **all 13** districts, a scale bar and a title.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati = district_counts[district_counts["province_code"] == "NP03"]
bagmati_slides = landslides_utm[landslides_utm.within(bagmati.union_all())]

fig, ax = plt.subplots(figsize=(10, 8))
bagmati.plot(ax=ax, color=bagmati["colour"], edgecolor="white", linewidth=0.8)
bagmati_slides.plot(ax=ax, color="#1a1a1a", edgecolor="white", markersize=25)
nudge_b = {"Bhaktapur": (14, -10)}                     # keeps it clear of Kathmandu
for _, row in bagmati.iterrows():
    spot = row.geometry.representative_point()
    ax.annotate(row["district"], xy=(spot.x, spot.y), xytext=nudge_b.get(row["district"], (0, 0)),
                textcoords="offset points", ha="center", fontsize=8, path_effects=halo)
ax.add_artist(ScaleBar(dx=1, units="m", location="lower left", box_alpha=0))
ax.set_title("Practice landslide records, Bagmati Province (made-up data)", loc="left")
ax.set_axis_off()
plt.show()
```

Zooming in on one province leaves room to name every district, though the three small Kathmandu Valley districts still crowd together; one nudge separates Bhaktapur from Kathmandu. The scale bar adapts by itself to a shorter round length, because the map now covers a smaller area.
</details>

### Review 2 · The seven provinces

Make a reference map of the provinces: join `province_names.csv` to `provinces_utm` (B6), colour each province with a different soft colour from `cmap="Pastel2"`, and label each with its **name** at its representative point. Add a scale bar and a north arrow.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
names = pd.read_csv(DATA / "province_names.csv")
named = provinces_utm.merge(names, on="province_code")

fig, ax = plt.subplots(figsize=(12, 7))
named.plot(ax=ax, column="province", cmap="Pastel2", edgecolor="white", linewidth=1)
for _, row in named.iterrows():
    spot = row.geometry.representative_point()
    ax.annotate(row["province"], xy=(spot.x, spot.y), ha="center", fontsize=10, fontweight="bold")
ax.add_artist(ScaleBar(dx=1, units="m", location="lower right", box_alpha=0))
ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction", ha="center",
            arrowprops={"facecolor": "black", "width": 3, "headwidth": 10})
ax.set_title("Provinces of Nepal", loc="left", fontsize=14)
ax.set_axis_off()
plt.show()
```

Provinces are categories, not amounts, so a **qualitative** palette such as `Pastel2` fits: different colours with no order, none louder than the others. Direct labels replace a legend.
</details>

### Review 3 · The Everest Base Camp trek

Read `everest_trek.csv`, make the stops a GeoDataFrame in UTM 45N, and join them into a line with `LineString` (B2). Plot the Solukhumbu district, the line and the stops, and label each stop with its name and elevation. Add a scale bar.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
from shapely.geometry import LineString

trek = pd.read_csv(DATA / "everest_trek.csv").sort_values("order")
stops = gpd.GeoDataFrame(trek, geometry=gpd.points_from_xy(trek["lon"], trek["lat"]), crs="EPSG:4326").to_crs(32645)
route = gpd.GeoSeries([LineString(stops.geometry.tolist())], crs=32645)

fig, ax = plt.subplots(figsize=(7, 8))
route.plot(ax=ax, color="#d7301f", linewidth=2)
stops.plot(ax=ax, color="black", markersize=20, zorder=3)
for _, row in stops.iterrows():
    ax.annotate(f"{row['place']} ({row['elevation_m']:,} m)", xy=(row.geometry.x, row.geometry.y),
                xytext=(6, 0), textcoords="offset points", fontsize=8, va="center", path_effects=halo)
ax.add_artist(ScaleBar(dx=1, units="m", location="lower right", box_alpha=0))
ax.set_title("Lukla to Everest Base Camp (approximate route)", loc="left")
ax.set_axis_off()
plt.show()
```

The trek is short enough that the scale bar shows a few kilometres. Labels to the right of each stop (`xytext=(6, 0)`) suit a route that runs mostly north. The district outline was left out to keep the map focused; add it with `district_counts[district_counts["district"] == "Solukhumbu"]` if you want the context.
</details>

## Recap

- Plan first: purpose, audience, medium and elements. Put every layer of the main map in one projected CRS (UTM 45N).
- Draw from the bottom up (fill, lines, points, labels), use `facecolor="none"` for outlines, and `zorder` when needed.
- Choose colours on purpose: light to dark for more, a neutral colour for zero, no rainbow, and the same colours across a report.
- Label with `representative_point()` and `ax.annotate`, with white halos and hand nudges. Add `ScaleBar(dx=1, units="m")`, a north arrow, a hand-built legend and an inset locator map.
- Remove clutter, add a title and a source note that flags made-up data, and save with `dpi=300, bbox_inches="tight"`. Then run the checklist.

### Words you learned

| Word | Meaning |
| --- | --- |
| purpose / audience / medium | What the map must show, who reads it, and where it appears |
| layer order | The sequence in which layers are drawn; later layers cover earlier ones |
| `zorder` | A number that sets drawing order directly; higher is on top |
| hex code | A colour written as `#rrggbb`, such as `#d7301f` |
| sequential / qualitative palette | Colours ordered light to dark for amounts / unordered colours for categories |
| `representative_point()` | A point guaranteed to lie inside a polygon, used for labels |
| halo | A thin outline around text that keeps it readable on any background |
| `iterrows()` | Loops over the rows of a table, one row at a time |
| scale bar | A bar showing a real distance on the map; needs a projected CRS |
| north arrow | A symbol showing the direction of north |
| meridian convergence | The angle between true north and the map's up direction in a projection |
| legend handle | A sample symbol with a label, such as a `Patch` or a `Line2D` |
| inset / locator map | A small map in a corner that shows where the main map is |
| dpi | Dots per inch: the resolution of a saved image; 300 for print |

## Next

**This is the end of Part B · Vector GIS.** You can now read, project, select, join, cut, count and map vector data. **Part C · Rasters begins with C1 · What a raster is:** grids of cells such as elevation models and satellite images, where every cell holds a value.

<!-- pygile-nav -->
**PyGILE Course** · lesson 21 of 21 · [Course home](../README.md)  
← Previous: [B10 · Counting points and density maps: where are the landslides?](B10_Counting_points_and_density_maps.ipynb) · You have reached the end of the course